# Rainfall Pipeline — Smoke Test

1. **Loader (bronze)** — downloads HDX CHIRPS subnational rainfall CSVs into `data/raw_rainfall/`.
2. **Transformer (silver)** — subsamples to monthly granularity and concatenates into a single Parquet at `data/clean_rainfall/rainfall_monthly.parquet`.

## 1. Loader — fetch raw CSVs

In [1]:
import sys
import os
from pathlib import Path

# Make the `libs/` package importable
project_root = str(Path(os.getcwd()).resolve())
if project_root not in sys.path:
    sys.path.append(project_root)

from libs.hdx_rainfall_loader import HDXRainfallLoader

loader = HDXRainfallLoader()

In [ ]:
results = loader.fetch_many(["YEM", "AFG", "NGA"])
results

In [ ]:
import pandas as pd

pd.read_csv(results["YEM"]).head()

## 2. Transformer — build the monthly Parquet

In [2]:
from libs.rainfall_transformer import RainfallTransformer

# Using fastparquet because pyarrow's DLLs misbehave on this Windows env
out_path = RainfallTransformer(engine="fastparquet").build()
out_path

2026-05-17 11:01:44,234 - rainfall_transformer - INFO - [AFG] Reading afg-rainfall-subnat-full.csv
2026-05-17 11:01:45,459 - rainfall_transformer - INFO - [AFG] Kept 235,552 monthly rows
2026-05-17 11:01:45,461 - rainfall_transformer - INFO - [NGA] Reading nga-rainfall-subnat-full.csv
2026-05-17 11:01:47,669 - rainfall_transformer - INFO - [NGA] Kept 435,744 monthly rows
2026-05-17 11:01:47,670 - rainfall_transformer - INFO - [YEM] Reading yem-rainfall-subnat-full.csv
2026-05-17 11:01:48,572 - rainfall_transformer - INFO - [YEM] Kept 184,960 monthly rows
2026-05-17 11:01:50,882 - rainfall_transformer - INFO - Wrote 856,256 rows (26.1 MB) -> C:\Users\jonas\Documents\master-data-science\progettone\HERO\rainfall\data\clean_rainfall\rainfall_monthly.parquet


WindowsPath('C:/Users/jonas/Documents/master-data-science/progettone/HERO/rainfall/data/clean_rainfall/rainfall_monthly.parquet')

In [4]:
import pandas as pd
df = pd.read_parquet(out_path, engine="fastparquet")
print(df.shape)
df.head()

(856256, 12)


,date,adm_level,adm_id,PCODE,r1h,r1h_avg,r3h,r3h_avg,rfq,r1q,r3q,ISO3
0,1981-01-21,1,900483,AF17,33.691124,32.502464,NaN,NaN,132.95471,103.16955,NaN,AFG
1,1981-02-21,1,900483,AF17,70.944250,54.722034,NaN,NaN,139.26776,127.16287,NaN,AFG
2,1981-03-21,1,900483,AF17,54.757675,72.452280,159.39305,159.67677,45.46830,77.15419,99.827710,AFG
3,1981-04-21,1,900483,AF17,64.486916,63.279540,190.18884,190.45384,84.86583,101.76828,99.864420,AFG
4,1981-05-21,1,900483,AF17,37.945393,52.429768,157.18999,188.16159,72.46708,74.77898,83.965965,AFG


In [5]:
# Sanity checks: only last-dekad rows, expected countries, row counts per country
print("Unique day-of-month:", df["date"].dt.day.unique())
print("Countries:", list(df["ISO3"].cat.categories))
df.groupby("ISO3", observed=True).size()

Unique day-of-month: [21]
Countries: ['AFG', 'NGA', 'YEM']


ISO3
AFG    235552
NGA    435744
YEM    184960
dtype: int64

## 3. Boundaries — fetch admin GeoJSON

In [1]:
from libs.hdx_boundaries_loader import HDXBoundariesLoader

boundaries = HDXBoundariesLoader().fetch_many(["YEM", "AFG", "NGA"])
boundaries

2026-05-17 18:10:12,584 - hdx_boundaries_loader - INFO - [YEM] Downloading yem_admin_boundaries.geojson.zip
2026-05-17 18:10:15,059 - hdx_boundaries_loader - INFO - [YEM] Saved -> C:\Users\jonas\Documents\master-data-science\progettone\HERO\rainfall\data\raw_boundaries\yem\yem_admin_boundaries.geojson.zip
2026-05-17 18:10:15,262 - hdx_boundaries_loader - INFO - [AFG] Downloading afg_admin_boundaries.geojson.zip
2026-05-17 18:10:17,016 - hdx_boundaries_loader - INFO - [AFG] Saved -> C:\Users\jonas\Documents\master-data-science\progettone\HERO\rainfall\data\raw_boundaries\afg\afg_admin_boundaries.geojson.zip
2026-05-17 18:10:17,192 - hdx_boundaries_loader - INFO - [NGA] Downloading nga_admin_boundaries.geojson.zip
2026-05-17 18:10:18,594 - hdx_boundaries_loader - INFO - [NGA] Saved -> C:\Users\jonas\Documents\master-data-science\progettone\HERO\rainfall\data\raw_boundaries\nga\nga_admin_boundaries.geojson.zip


{'YEM': WindowsPath('C:/Users/jonas/Documents/master-data-science/progettone/HERO/rainfall/data/raw_boundaries/yem/yem_admin_boundaries.geojson.zip'),
 'AFG': WindowsPath('C:/Users/jonas/Documents/master-data-science/progettone/HERO/rainfall/data/raw_boundaries/afg/afg_admin_boundaries.geojson.zip'),
 'NGA': WindowsPath('C:/Users/jonas/Documents/master-data-science/progettone/HERO/rainfall/data/raw_boundaries/nga/nga_admin_boundaries.geojson.zip')}